In [33]:
import pandas as pd

data = pd.read_csv("../data/risks.csv")

In [34]:
#definitions
risk_priority = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

data["risk_priority"] = data["risk_level"].map(risk_priority)

strategies = {
    "Conservative": {
        "max_risk": 1,
        "min_delay": 71,
        "sort_by": ["risk_priority", "op_reliab"],
        "ascending": [True, False]
    },
    "Balanced": {
        "max_risk": 3,
        "min_delay": 41,
        "max_delay": 70,
        "sort_by": ["startup_cost", "month_op_cost", "risk_priority"],
        "ascending": [True, True, True]
    },
    "Accelerated": {
        "max_risk": 2,
        "max_delay": 40,
        "sort_by": ["rec_perf_ratio", "op_reliab"],
        "ascending": [False, False]
    }
}

In [35]:
#selection function
def select_sites(data, strategy_name, n_sites):
    strategy = strategies[strategy_name]
    eligible_sites = data.copy()
    eligible_sites = eligible_sites[
        eligible_sites["risk_priority"] <= strategy["max_risk"]
    ]

    if "min_delay" in strategy:
        eligible_sites = eligible_sites[
            eligible_sites["act_delay_days"] >= strategy["min_delay"]
        ]

    if "max_delay" in strategy:
        eligible_sites = eligible_sites[
            eligible_sites["act_delay_days"] <= strategy["max_delay"]
        ]

    selected_sites = (
        eligible_sites
        .sort_values(
            by=strategy["sort_by"],
            ascending=strategy["ascending"]
        )
        .head(n_sites)
        .copy()
    )

    return selected_sites

In [42]:
#controlpoint
n_sites = 30

selected_by_strategy = {
    name: select_sites(data, name, n_sites)
    for name in strategies
}

for name, sites in selected_by_strategy.items():
    print(f"{name}: {len(sites)} of {n_sites} sites selected")

Conservative: 23 of 30 sites selected
Balanced: 30 of 30 sites selected
Accelerated: 30 of 30 sites selected


In [43]:
strategy_results = []

for strategy_name, sites in selected_by_strategy.items():
    strategy_results.append({
        "Strategy": strategy_name,
        "Sites selected": len(sites),
        "Median activation delay (days)": sites["act_delay_days"].median(),
        "Median recruitment performance": sites["rec_perf_ratio"].median(),
        "Median operational reliability": sites["op_reliab"].median(),
        "Total startup cost": sites["startup_cost"].sum(),
        "Total monthly operating cost": sites["month_op_cost"].sum(),
        "High-risk sites": (sites["risk_level"] == "High").sum()
    })

results = pd.DataFrame(strategy_results)

results

,Strategy,Sites selected,Median activation delay (days),Median recruitment performance,Median operational reliability,Total startup cost,Total monthly operating cost,High-risk sites
0,Conservative,23,84.0,1.020,0.890,194866,67190,0
1,Balanced,30,57.5,0.975,0.700,252123,89559,3
2,Accelerated,30,27.0,1.080,0.765,367875,113797,0
